# 📞 Qwen voice agent on the phone — LiveKit + Google Colab

A caller dials a phone number → **LiveKit Cloud** answers and puts the call in a room →
this notebook's agent joins and talks:

```
caller ──phone──▶ LiveKit Cloud (free number + SIP) ──WebRTC──▶ agent (this Colab GPU)
                                                                  │  Silero VAD (turn taking)
                                                                  │  Qwen3-ASR-0.6B   (speech → text)
                                                                  │  Qwen2.5-7B via Ollama (thinking + tools)
                                                                  └  Qwen3-TTS-0.6B   (text → speech, voice "Ryan")
```

The agent only makes **outbound** connections to LiveKit, so Colab needs no open ports.

**Before you start** (one-time, ~5 minutes — details in the README next to this notebook):
1. Create a free account at **https://cloud.livekit.io** → new project → *Settings → API keys*: note the
   **URL** (`wss://…livekit.cloud`), **API key**, **API secret**.
2. In Colab, open the 🔑 **Secrets** panel (left sidebar) and add `LIVEKIT_URL`, `LIVEKIT_API_KEY`,
   `LIVEKIT_API_SECRET`, and toggle *Notebook access* on for each.
3. *Runtime → Change runtime type → **T4 GPU***.
4. In the LiveKit dashboard: *Telephony → Phone numbers* → get the free US number, and create a
   **dispatch rule** for it (*Individual* rooms). You can also test without a phone using the
   *Agents Playground* (step 8).

Run the cells top to bottom. First run downloads ~10 GB of models (≈10 min).

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install packages
`qwen-asr` and `qwen-tts` pin different `transformers` patch versions (4.57.6 vs 4.57.3), so `qwen-tts`
is installed with `--no-deps` on top of `qwen-asr` and its few extra dependencies are added by hand.

In [ ]:
!apt-get -qq install -y sox libsox-fmt-all zstd > /dev/null
!pip install -q -U qwen-asr
!pip install -q -U --no-deps qwen-tts
!pip install -q onnxruntime einops torchaudio soundfile librosa
!pip install -q "livekit-agents[openai,silero]~=1.9"
!pip list 2>/dev/null | grep -iE "^(qwen-asr|qwen-tts|transformers|livekit-agents|torch) "

## 3. Install Ollama and pull the Qwen LLM (~4.7 GB)

In [ ]:
import subprocess, time, urllib.request
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2); break
    except Exception:
        time.sleep(1)
LLM_MODEL = "qwen2.5:7b"   # smaller/faster: "qwen2.5:3b"
!ollama pull {LLM_MODEL}

## 4. Write the agent files
These are the same files as in the `voice_agent_colab/` folder.

In [ ]:
%%writefile model_server.py
"""Qwen speech model server: loads Qwen3-ASR and Qwen3-TTS ONCE on the GPU and
serves them over localhost, so every LiveKit call shares the same models.

  GET  /health                      -> {"ok": true, "asr": ..., "tts": ..., "speaker": ...}
  POST /asr?sr=16000                body: raw little-endian int16 mono PCM
                                    -> {"text": "...", "language": "English"}
  POST /tts                         body: {"text": "...", "speaker"?: "Ryan", "language"?: "English"}
                                    -> raw int16 mono PCM at 24 kHz (header X-Sample-Rate)

Run:  python model_server.py            (real models, needs a CUDA GPU)
      python model_server.py --fake     (no models: canned text + a tone, for plumbing tests)

Settings (env vars): ASR_MODEL, TTS_MODEL, TTS_SPEAKER, ASR_LANGUAGE, TTS_LANGUAGE,
MODEL_SERVER_PORT (default 8765).
"""

from __future__ import annotations

import argparse
import json
import os
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlsplit

import numpy as np

OUT_SR = 24_000  # the TTS plugin declares this rate; we resample to it
ASR_MODEL = os.environ.get("ASR_MODEL", "Qwen/Qwen3-ASR-0.6B")
TTS_MODEL = os.environ.get("TTS_MODEL", "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice")
TTS_SPEAKER = os.environ.get("TTS_SPEAKER", "Ryan")
ASR_LANGUAGE = os.environ.get("ASR_LANGUAGE", "English")  # "auto" = detect
TTS_LANGUAGE = os.environ.get("TTS_LANGUAGE", "English")  # "Auto" = detect
MAX_BODY = 20 * 1024 * 1024


class Models:
    def __init__(self, fake: bool):
        self.fake = fake
        self.asr = self.tts = None
        self.asr_lock = threading.Lock()  # one GPU job per model at a time
        self.tts_lock = threading.Lock()
        if fake:
            return
        import torch
        from qwen_asr import Qwen3ASRModel
        from qwen_tts import Qwen3TTSModel

        if not torch.cuda.is_available():
            raise SystemExit("No CUDA GPU. In Colab: Runtime > Change runtime type > T4 GPU.")
        major, _ = torch.cuda.get_device_capability()
        dtype = torch.bfloat16 if major >= 8 else torch.float16  # T4 (sm75) has no bf16
        attn = "sdpa"
        try:
            import flash_attn  # noqa: F401
            if major >= 8:
                attn = "flash_attention_2"
        except ImportError:
            pass
        print(f"[models] GPU {torch.cuda.get_device_name(0)}  dtype={dtype}  attn={attn}", flush=True)

        t = time.time()
        self.asr = Qwen3ASRModel.from_pretrained(
            ASR_MODEL, dtype=dtype, device_map="cuda:0", attn_implementation=attn,
            max_inference_batch_size=4, max_new_tokens=256)
        print(f"[models] ASR {ASR_MODEL} loaded in {time.time() - t:.0f}s", flush=True)

        t = time.time()
        self.tts = Qwen3TTSModel.from_pretrained(
            TTS_MODEL, dtype=dtype, device_map="cuda:0", attn_implementation=attn)
        speakers = self.tts.get_supported_speakers() or []
        print(f"[models] TTS {TTS_MODEL} loaded in {time.time() - t:.0f}s; speakers={speakers}", flush=True)
        self.warmup()

    def warmup(self):
        t = time.time()
        pcm, _ = self.synthesize("Hello, I'm ready.")
        self.transcribe(pcm, OUT_SR)
        print(f"[models] warm-up round trip {time.time() - t:.1f}s", flush=True)

    # -- inference -------------------------------------------------------------
    def transcribe(self, pcm: np.ndarray, sr: int) -> dict:
        if self.fake:
            return {"text": "what time is it" if len(pcm) > sr // 4 else "", "language": "English"}
        audio = pcm.astype(np.float32) / 32768.0
        lang = None if ASR_LANGUAGE.lower() == "auto" else ASR_LANGUAGE
        with self.asr_lock:
            res = self.asr.transcribe(audio=(audio, sr), language=lang)[0]
        return {"text": (res.text or "").strip(), "language": res.language}

    def synthesize(self, text: str, speaker: str | None = None,
                   language: str | None = None) -> tuple[np.ndarray, int]:
        if self.fake:
            t = np.arange(int(OUT_SR * min(3.0, 0.06 * len(text) + 0.3))) / OUT_SR
            return (np.sin(2 * np.pi * 440 * t) * 8000).astype(np.int16), OUT_SR
        with self.tts_lock:
            wavs, sr = self.tts.generate_custom_voice(
                text=text, speaker=speaker or TTS_SPEAKER, language=language or TTS_LANGUAGE)
        wav = np.asarray(wavs[0], dtype=np.float32)
        if sr != OUT_SR:
            import librosa
            wav = librosa.resample(wav, orig_sr=sr, target_sr=OUT_SR)
        return (np.clip(wav, -1, 1) * 32767).astype(np.int16), OUT_SR


def make_handler(models: Models):
    class H(BaseHTTPRequestHandler):
        def _send(self, code, body, ctype="application/json", headers=None):
            data = body if isinstance(body, bytes) else json.dumps(body).encode()
            self.send_response(code)
            self.send_header("Content-Type", ctype)
            self.send_header("Content-Length", str(len(data)))
            for k, v in (headers or {}).items():
                self.send_header(k, v)
            self.end_headers()
            self.wfile.write(data)

        def _body(self) -> bytes:
            n = int(self.headers.get("Content-Length") or 0)
            if n > MAX_BODY:
                raise ValueError("body too large")
            return self.rfile.read(n) if n else b""

        def do_GET(self):
            if urlsplit(self.path).path == "/health":
                return self._send(200, {"ok": True, "fake": models.fake, "asr": ASR_MODEL,
                                        "tts": TTS_MODEL, "speaker": TTS_SPEAKER})
            self._send(404, {"error": "not found"})

        def do_POST(self):
            parts = urlsplit(self.path)
            try:
                body = self._body()
                if parts.path == "/asr":
                    sr = int(parse_qs(parts.query).get("sr", ["16000"])[0])
                    pcm = np.frombuffer(body, dtype=np.int16)
                    t = time.time()
                    out = models.transcribe(pcm, sr)
                    print(f"[asr {time.time() - t:.2f}s, {len(pcm) / sr:.1f}s audio] {out['text']!r}", flush=True)
                    return self._send(200, out)
                if parts.path == "/tts":
                    req = json.loads(body or b"{}")
                    text = str(req.get("text", "")).strip()
                    if not text:
                        return self._send(400, {"error": "empty text"})
                    t = time.time()
                    pcm, sr = models.synthesize(text, req.get("speaker"), req.get("language"))
                    print(f"[tts {time.time() - t:.2f}s -> {len(pcm) / sr:.1f}s audio] {text!r}", flush=True)
                    return self._send(200, pcm.tobytes(), "application/octet-stream",
                                      {"X-Sample-Rate": str(sr)})
            except Exception as e:  # noqa: BLE001 - report to the caller
                print(f"[error] {parts.path}: {e}", flush=True)
                return self._send(500, {"error": str(e)})
            self._send(404, {"error": "not found"})

        def log_message(self, *a):
            pass

    return H


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--fake", action="store_true", help="no models; for testing the plumbing")
    ap.add_argument("--port", type=int, default=int(os.environ.get("MODEL_SERVER_PORT", "8765")))
    args = ap.parse_args()
    models = Models(fake=args.fake)
    srv = ThreadingHTTPServer(("127.0.0.1", args.port), make_handler(models))
    print(f"[models] serving on http://127.0.0.1:{args.port}  fake={args.fake}", flush=True)
    srv.serve_forever()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile qwen_plugins.py
"""LiveKit Agents STT/TTS plugins backed by the local Qwen model server.

Both are "non-streaming" in LiveKit terms:
  * QwenSTT gets one finished utterance at a time. AgentSession wraps it with the
    Silero VAD (stt.StreamAdapter) automatically, so speech is cut into turns.
  * QwenTTS synthesises one sentence at a time. AgentSession wraps it with a
    sentence tokenizer (tts.StreamAdapter), so the caller hears the first sentence
    while the LLM is still writing the next.
"""

from __future__ import annotations

import os
import uuid

import aiohttp
import numpy as np
from livekit import rtc
from livekit.agents import (
    DEFAULT_API_CONNECT_OPTIONS,
    APIConnectionError,
    APIConnectOptions,
    APIStatusError,
    stt,
    tts,
)
from livekit.agents.utils import AudioBuffer

SERVER = os.environ.get("MODEL_SERVER_URL", "http://127.0.0.1:8765")
TTS_SAMPLE_RATE = 24_000  # model_server.OUT_SR


class _HttpMixin:
    _session: aiohttp.ClientSession | None = None

    def _http(self) -> aiohttp.ClientSession:
        if self._session is None or self._session.closed:
            self._session = aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=60))
        return self._session

    async def aclose(self) -> None:
        if self._session and not self._session.closed:
            await self._session.close()


class QwenSTT(_HttpMixin, stt.STT):
    def __init__(self, server_url: str = SERVER, language: str = "en"):
        super().__init__(capabilities=stt.STTCapabilities(streaming=False, interim_results=False))
        self.server_url = server_url.rstrip("/")
        self.language = language

    @property
    def model(self) -> str:
        return "qwen3-asr"

    @property
    def provider(self) -> str:
        return "local"

    async def _recognize_impl(self, buffer: AudioBuffer, *, language=None,
                              conn_options: APIConnectOptions = DEFAULT_API_CONNECT_OPTIONS) -> stt.SpeechEvent:
        frame = rtc.combine_audio_frames(buffer)
        pcm = np.frombuffer(frame.data, dtype=np.int16)
        if frame.num_channels > 1:
            pcm = pcm.reshape(-1, frame.num_channels).mean(axis=1).astype(np.int16)
        try:
            async with self._http().post(f"{self.server_url}/asr?sr={frame.sample_rate}",
                                         data=pcm.tobytes()) as r:
                if r.status != 200:
                    raise APIStatusError(f"ASR server error: {await r.text()}", status_code=r.status)
                out = await r.json()
        except aiohttp.ClientError as e:
            raise APIConnectionError(f"ASR server unreachable: {e}") from e
        return stt.SpeechEvent(
            type=stt.SpeechEventType.FINAL_TRANSCRIPT,
            request_id=uuid.uuid4().hex,
            alternatives=[stt.SpeechData(language=self.language, text=out.get("text", ""))],
        )


class QwenTTS(_HttpMixin, tts.TTS):
    def __init__(self, server_url: str = SERVER, speaker: str | None = None, language: str | None = None):
        super().__init__(capabilities=tts.TTSCapabilities(streaming=False),
                         sample_rate=TTS_SAMPLE_RATE, num_channels=1)
        self.server_url = server_url.rstrip("/")
        self.speaker = speaker
        self.language = language

    @property
    def model(self) -> str:
        return "qwen3-tts"

    @property
    def provider(self) -> str:
        return "local"

    def synthesize(self, text: str, *,
                   conn_options: APIConnectOptions = DEFAULT_API_CONNECT_OPTIONS) -> tts.ChunkedStream:
        return _QwenChunkedStream(tts=self, input_text=text, conn_options=conn_options)


class _QwenChunkedStream(tts.ChunkedStream):
    async def _run(self, output_emitter: tts.AudioEmitter) -> None:
        t: QwenTTS = self._tts  # type: ignore[assignment]
        body = {"text": self.input_text}
        if t.speaker:
            body["speaker"] = t.speaker
        if t.language:
            body["language"] = t.language
        try:
            async with t._http().post(f"{t.server_url}/tts", json=body) as r:
                if r.status != 200:
                    raise APIStatusError(f"TTS server error: {await r.text()}", status_code=r.status)
                sr = int(r.headers.get("X-Sample-Rate", TTS_SAMPLE_RATE))
                output_emitter.initialize(request_id=uuid.uuid4().hex, sample_rate=sr,
                                          num_channels=1, mime_type="audio/pcm")
                async for chunk in r.content.iter_chunked(8192):
                    output_emitter.push(chunk)
        except aiohttp.ClientError as e:
            raise APIConnectionError(f"TTS server unreachable: {e}") from e
        output_emitter.flush()

In [ ]:
%%writefile agent.py
"""LiveKit phone/voice agent: Qwen3-ASR -> Qwen LLM (Ollama) -> Qwen3-TTS.

Run (after model_server.py and Ollama are up, and LIVEKIT_* env vars are set):
    python agent.py start        # production mode, waits for calls
    python agent.py dev          # verbose logs
Env: LIVEKIT_URL, LIVEKIT_API_KEY, LIVEKIT_API_SECRET, LLM_MODEL (default qwen2.5:7b),
     OLLAMA_URL (default http://127.0.0.1:11434/v1), AGENT_NAME (optional).
"""

from __future__ import annotations

import html
import logging
import os
import re
import urllib.parse
from datetime import datetime, timezone

import aiohttp
from livekit.agents import Agent, AgentServer, AgentSession, JobContext, JobProcess, RunContext, cli, function_tool
from livekit.plugins import openai, silero

from qwen_plugins import QwenSTT, QwenTTS

log = logging.getLogger("qwen-voice-agent")
LLM_MODEL = os.environ.get("LLM_MODEL", "qwen2.5:7b")
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434/v1")
AGENT_NAME = os.environ.get("AGENT_NAME", "")  # empty = join every new room automatically

INSTRUCTIONS = """You are a friendly, concise voice assistant answering a phone call.
Speak naturally in short sentences; one to three sentences per reply unless asked for more.
Never use markdown, lists, emoji, or read out URLs. Spell numbers the way people say them.
If you need current information (news, prices, weather, recent facts), call web_search first
and answer only from what it returns. If you are unsure, say so. If the caller says goodbye,
say a short goodbye."""


class VoiceAssistant(Agent):
    def __init__(self) -> None:
        super().__init__(instructions=INSTRUCTIONS)

    @function_tool
    async def get_current_time(self, context: RunContext) -> str:
        """Get the current date and time (UTC)."""
        return datetime.now(timezone.utc).strftime("%A %d %B %Y, %H:%M UTC")

    @function_tool
    async def web_search(self, context: RunContext, query: str) -> str:
        """Search the web for current information and return the top results.

        Args:
            query: what to search for
        """
        try:
            async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=10)) as s:
                async with s.post("https://lite.duckduckgo.com/lite/", data={"q": query},
                                  headers={"User-Agent": "Mozilla/5.0 voice-agent/0.1"}) as r:
                    page = await r.text()
        except Exception as e:  # noqa: BLE001
            return f"Search failed: {e}"
        titles = re.findall(r"class=['\"]result-link['\"][^>]*>(.*?)</a>", page, re.S)
        snippets = re.findall(r"class=['\"]result-snippet['\"][^>]*>(.*?)</td>", page, re.S)
        clean = lambda s: html.unescape(re.sub(r"<[^>]+>", "", s)).strip()  # noqa: E731
        results = [f"{clean(t)}: {clean(sn)}" for t, sn in zip(titles, snippets)][:4]
        return "\n".join(results) or "No results found."


def prewarm(proc: JobProcess) -> None:
    proc.userdata["vad"] = silero.VAD.load()


# Colab has only 2 CPUs and the GPU models keep them busy; LiveKit's default
# (stop accepting calls above 70% CPU) would make the agent look "unavailable".
server = AgentServer(setup_fnc=prewarm, num_idle_processes=1,
                     load_threshold=float(os.environ.get("LOAD_THRESHOLD", "inf")))


@server.rtc_session(agent_name=AGENT_NAME)
async def entrypoint(ctx: JobContext) -> None:
    session = AgentSession(
        vad=ctx.proc.userdata["vad"],
        stt=QwenSTT(),
        llm=openai.LLM.with_ollama(model=LLM_MODEL, base_url=OLLAMA_URL, temperature=0.4),
        tts=QwenTTS(),
        min_endpointing_delay=0.4,
    )

    @session.on("user_input_transcribed")
    def _heard(ev):
        if getattr(ev, "is_final", True):
            log.info("caller: %s", ev.transcript)

    await session.start(agent=VoiceAssistant(), room=ctx.room)
    await session.generate_reply(instructions="Greet the caller briefly and ask how you can help.")


if __name__ == "__main__":
    cli.run_app(server)

In [ ]:
%%writefile smoke_test.py
"""Plumbing test without LiveKit Cloud or a phone:
  1. STT plugin -> model server /asr
  2. TTS plugin -> model server /tts (audio frames come back)
  3. The agent's LLM + tools via AgentSession in text-only mode (needs Ollama)

Usage:  python smoke_test.py            (model server must be running, real or --fake)
        LLM_MODEL=qwen2.5:3b OLLAMA_URL=http://127.0.0.1:11434/v1 python smoke_test.py
"""

import asyncio
import os
import time

import numpy as np
from livekit import rtc
from livekit.agents import AgentSession
from livekit.plugins import openai

from qwen_plugins import QwenSTT, QwenTTS


async def main() -> None:
    sr = 48_000
    tone = (np.sin(2 * np.pi * 220 * np.arange(sr) / sr) * 6000).astype(np.int16)
    frame = rtc.AudioFrame(tone.tobytes(), sample_rate=sr, num_channels=1, samples_per_channel=sr)

    stt_ = QwenSTT()
    t = time.time()
    ev = await stt_.recognize([frame])
    print(f"STT ({time.time() - t:.2f}s): {ev.type.value} -> {ev.alternatives[0].text!r}")

    tts_ = QwenTTS()
    t = time.time()
    frames = [f async for f in tts_.synthesize("Hello! This is a smoke test of the voice pipeline.")]
    secs = sum(f.frame.samples_per_channel for f in frames) / tts_.sample_rate
    print(f"TTS ({time.time() - t:.2f}s): {len(frames)} frames, {secs:.2f}s of audio")

    # Optional: round-trip real TTS audio back through ASR (meaningful with real models)
    audio = rtc.combine_audio_frames([f.frame for f in frames])
    ev = await stt_.recognize([audio])
    print(f"TTS->ASR round trip: {ev.alternatives[0].text!r}")
    await stt_.aclose()
    await tts_.aclose()

    if os.environ.get("SKIP_LLM"):
        return
    from agent import LLM_MODEL, OLLAMA_URL, VoiceAssistant
    async with openai.LLM.with_ollama(model=LLM_MODEL, base_url=OLLAMA_URL) as llm, \
            AgentSession(llm=llm) as session:
        await session.start(VoiceAssistant())
        for q in ["What's the date today?", "Search the web: what is the latest Python release?"]:
            t = time.time()
            result = await session.run(user_input=q)
            calls = [e.item.name for e in result.events if e.type == "function_call"]
            reply = [e.item.text_content for e in result.events if e.type == "message"]
            print(f"LLM ({time.time() - t:.1f}s) {q!r}\n  tools={calls}\n  reply={reply[-1] if reply else None!r}")


if __name__ == "__main__":
    asyncio.run(main())

## 5. Start the Qwen model server
Loads Qwen3-ASR and Qwen3-TTS once on the GPU (first run downloads them). Wait for **ready**.
Change the voice with `TTS_SPEAKER` (English: `Ryan`, `Aiden`; others: `Vivian`, `Serena`, `Uncle_Fu`,
`Dylan`, `Eric`, `Ono_Anna`, `Sohee`).

In [ ]:
import os, subprocess, time, json, urllib.request
os.environ.update({"TTS_SPEAKER": "Ryan", "ASR_LANGUAGE": "English", "TTS_LANGUAGE": "English"})
model_server = subprocess.Popen(["python", "model_server.py"], stdout=open("model_server.log", "w"),
                                stderr=subprocess.STDOUT)
for i in range(180):
    try:
        print("ready:", json.load(urllib.request.urlopen("http://127.0.0.1:8765/health", timeout=2)))
        break
    except Exception:
        if model_server.poll() is not None:
            print("model server exited; log:"); print(open("model_server.log").read()[-3000:]); break
        time.sleep(5)
!tail -n 6 model_server.log

## 6. Listen to the voice and test the whole pipeline

In [ ]:
import numpy as np, urllib.request, json
from IPython.display import Audio, display
req = urllib.request.Request("http://127.0.0.1:8765/tts", method="POST",
      data=json.dumps({"text": "Hi! I'm your Qwen voice assistant. How can I help you today?"}).encode())
with urllib.request.urlopen(req) as r:
    pcm = np.frombuffer(r.read(), dtype=np.int16); sr = int(r.headers["X-Sample-Rate"])
display(Audio(pcm.astype(np.float32) / 32768, rate=sr))

In [ ]:
import os
os.environ.update({"LLM_MODEL": LLM_MODEL, "OLLAMA_URL": "http://127.0.0.1:11434/v1"})
!python smoke_test.py 2>&1 | grep -vE "^(DEBUG|INFO)" | tail -n 15
!tail -n 4 model_server.log

## 7. Start the phone agent
Reads your LiveKit credentials from Colab Secrets and connects to LiveKit Cloud.
Look for `registered worker` in the log.

In [ ]:
import os, subprocess, time
from google.colab import userdata
for k in ["LIVEKIT_URL", "LIVEKIT_API_KEY", "LIVEKIT_API_SECRET"]:
    os.environ[k] = userdata.get(k)
agent_proc = subprocess.Popen(["python", "agent.py", "start"], stdout=open("agent.log", "w"),
                              stderr=subprocess.STDOUT)
time.sleep(20)
!grep -iE "registered|error|warn" agent.log | tail -n 5

## 8. Talk to it
* **Phone:** call the LiveKit number from your mobile. The first reply can take a few seconds while
  everything warms up.
* **Browser (no phone needed):** open https://agents-playground.livekit.io, sign in with the same
  LiveKit project, *Connect*, and allow the microphone.

Run the next cell any time to see what the caller said and what the models did.

In [ ]:
!grep -E "caller:|error|Error" agent.log | tail -n 20
!grep -E "^\[(asr|tts)" model_server.log | tail -n 12

## 9. Stop everything
(Colab also stops it all when the runtime disconnects.)

In [ ]:
for p in ["agent_proc", "model_server", "ollama"]:
    if p in globals():
        globals()[p].terminate()
print("stopped")